# 02 · Matrix Multiplication as a Machine

Companion to **Chapter 2**. The one rule: **a matmul consumes one dimension.**
`(m,k) @ (k,n) -> (m,n)`. The two `k`s meet, get multiplied-and-summed, and vanish.

In [ ]:
import torch, math, time
import torch.nn as nn
torch.manual_seed(0)

## 1 · One output cell at a time

In [ ]:
A = torch.randint(-3, 4, (4, 3))
B = torch.randint(-3, 4, (3, 5))
C = A @ B
print("A", tuple(A.shape), " B", tuple(B.shape), " ->  C", tuple(C.shape))

i, j = 2, 3
terms = " + ".join(f"{A[i,t]}x{B[t,j]}" for t in range(3))
print(f"\nC[{i},{j}] = dot(A[{i},:], B[:,{j}])")
print(f"        = {terms}")
print(f"        = {C[i,j].item()}")
assert C[i, j] == (A[i, :] * B[:, j]).sum()
print("\nThe shared dim k=3 is summed away. It is NOT in the output shape.")

## 2 · Batched matmul — leading axes ride along

`@` operates on the **last two axes only**. Everything before is a batch index.

In [ ]:
cases = [
    ("(k,) @ (k,)",              torch.randn(5),          torch.randn(5)),
    ("(m,k) @ (k,)",             torch.randn(4, 5),       torch.randn(5)),
    ("(m,k) @ (k,n)",            torch.randn(4, 5),       torch.randn(5, 7)),
    ("(b,m,k) @ (k,n)",          torch.randn(8, 4, 5),    torch.randn(5, 7)),
    ("(b,m,k) @ (b,k,n)",        torch.randn(8, 4, 5),    torch.randn(8, 5, 7)),
    ("(b,h,m,k) @ (b,h,k,n)",    torch.randn(2, 8, 4, 5), torch.randn(2, 8, 5, 7)),
    ("(2,1,m,k) @ (1,6,k,n)",    torch.randn(2, 1, 4, 5), torch.randn(1, 6, 5, 7)),
]
for label, a, b in cases:
    print(f"{label:26} {str(tuple(a.shape)):14} @ {str(tuple(b.shape)):14} "
          f"-> {tuple((a @ b).shape)}")

print("\nThe second-to-last row IS multi-head attention:")
print("  (B, n_head, T, d_head) @ (B, n_head, d_head, T) -> (B, n_head, T, T)")

# and one that fails
try:
    torch.randn(8, 4, 3) @ torch.randn(5, 3)
except RuntimeError as e:
    print(f"\n(8,4,3) @ (5,3) -> RuntimeError: inner dims 3 vs 5 do not match")

## 3 · `nn.Linear` stores its weight TRANSPOSED

In [ ]:
lin = nn.Linear(512, 2048, bias=False)
print(f"nn.Linear(512, 2048).weight.shape = {tuple(lin.weight.shape)}   <- OUT first!")

x = torch.randn(8, 128, 512)
print(f"input  {tuple(x.shape)}")
print(f"output {tuple(lin(x).shape)}   <- only the LAST axis changed")
assert torch.allclose(lin(x), x @ lin.weight.T, atol=1e-5)
print("\nlin(x) == x @ W.T  ✓")
print(f"parameters: 512 x 2048 = {512*2048:,}  (independent of B and T)")

## 4 · Broadcasting — and the bug that doesn't crash

In [ ]:
def try_broadcast(a_shape, b_shape):
    try:
        out = tuple((torch.zeros(a_shape) + torch.zeros(b_shape)).shape)
        return f"-> {out}"
    except RuntimeError:
        return "-> ERROR (incompatible)"

for a, b in [((3,1,4),(5,4)), ((8,1,6,1),(7,1,5)), ((2,3),(4,3)),
             ((10,4),(4,)), ((10,4),(10,)), ((10,1),(1,7))]:
    print(f"{str(a):14} + {str(b):12} {try_broadcast(a, b)}")

print("\n(10,4) + (10,) FAILS but (10,4) + (4,) works: shapes align from the RIGHT.")
print("To add one number per ROW you need (10,1):", 
      tuple((torch.zeros(10,4) + torch.zeros(10)[:,None]).shape))

In [ ]:
# THE classic silent catastrophe
pred   = torch.randn(32, 1)
target = torch.randn(32)

bad  = ((pred - target) ** 2).mean()
good = ((pred.squeeze(-1) - target) ** 2).mean()
print(f"shape of (pred - target): {tuple((pred-target).shape)}  <- 32x32, not 32!")
print(f"buggy loss:   {bad.item():.4f}   (mean over 1024 all-pairs differences)")
print(f"correct loss: {good.item():.4f}")
print("\nNo error. Wrong loss. Garbage training.")
print("ALWAYS: assert pred.shape == target.shape  before a loss.")

## 5 · einsum — say what you mean

**Any letter in the inputs but not the output is summed over.**

In [ ]:
A2, B2 = torch.randn(4, 3), torch.randn(3, 5)
v, w = torch.randn(4), torch.randn(5)

ops = [
    ("'ik,kj->ij'  matmul (k summed)",  torch.einsum('ik,kj->ij', A2, B2)),
    ("'ij->ji'     transpose",          torch.einsum('ij->ji', A2)),
    ("'ij->'       sum everything",     torch.einsum('ij->', A2)),
    ("'ij->i'      row sums",           torch.einsum('ij->i', A2)),
    ("'i,i->'      dot product",        torch.einsum('i,i->', v, v)),
    ("'i,j->ij'    outer product",      torch.einsum('i,j->ij', v, w)),
]
for label, r in ops:
    print(f"{label:36} -> {tuple(r.shape)}")
assert torch.allclose(torch.einsum('ik,kj->ij', A2, B2), A2 @ B2, atol=1e-5)

In [ ]:
# Attention, written two ways -- identical results
q, k, v_ = (torch.randn(2, 8, 100, 64) for _ in range(3))

s_ein = torch.einsum('bhqd,bhkd->bhqk', q, k) / 64**0.5   # d is summed away
s_mat = (q @ k.transpose(-2, -1)) / 64**0.5
assert torch.allclose(s_ein, s_mat, atol=1e-4)

a = s_ein.softmax(-1)
o_ein = torch.einsum('bhqk,bhkd->bhqd', a, v_)            # now k is summed away
o_mat = a @ v_
assert torch.allclose(o_ein, o_mat, atol=1e-4)

print(f"scores {tuple(s_ein.shape)}   output {tuple(o_ein.shape)}")
print("einsum == @  ✓")
print("\n'bhqd,bhkd->bhqk' : d vanishes -> the dot product")
print("'bhqk,bhkd->bhqd' : k vanishes -> the weighted average")
print("\nThe einsum version is impossible to get backwards. That is its value.")

## 6 · Exercise 2.2 — matmul three ways

In [ ]:
def matmul_loops(A, B):
    m, k = A.shape; _, n = B.shape
    C = torch.zeros(m, n)
    for i in range(m):
        for j in range(n):
            C[i, j] = (A[i, :] * B[:, j]).sum()
    return C

def matmul_broadcast(A, B):
    # (m,k,1) * (1,k,n) -> (m,k,n), then sum over k
    return (A[:, :, None] * B[None, :, :]).sum(dim=1)

def matmul_einsum(A, B):
    return torch.einsum('ik,kj->ij', A, B)

N = 128
A3, B3 = torch.randn(N, N), torch.randn(N, N)
ref = A3 @ B3

print(f"{'method':>18} {'time':>10} {'matches':>9}")
for f in (matmul_loops, matmul_broadcast, matmul_einsum):
    t0 = time.perf_counter(); got = f(A3, B3); dt = time.perf_counter() - t0
    ok = torch.allclose(got, ref, atol=1e-3)
    print(f"{f.__name__:>18} {dt*1000:>9.1f}ms {str(ok):>9}")
    assert ok
t0 = time.perf_counter(); A3 @ B3; dt = time.perf_counter() - t0
print(f"{'A @ B':>18} {dt*1000:>9.1f}ms {'True':>9}")

inter = N * N * N * 4 / 1024**2
print(f"\nNote matmul_broadcast materialises an ({N},{N},{N}) intermediate = {inter:.0f} MB.")
print("'Materialising a big intermediate you immediately reduce away' is exactly")
print("the problem FlashAttention solves, at a much larger scale. -> Chapter 17.")

## 7 · Exercise 2.4 — FLOP counting

In [ ]:
print("Rule: training FLOPs ~ 6 * N * D   (2N forward, 4N backward)\n")
print(f"{'model':>14} {'params':>9} {'tokens':>9} {'FLOPs':>12} {'H100-days @40%':>16}")
for name, N_, D_ in [("GPT-3", 175e9, 300e9), ("Chinchilla", 70e9, 1.4e12),
                     ("Llama-3-8B", 8e9, 15e12), ("7B on 2T", 7e9, 2e12)]:
    flops = 6 * N_ * D_
    days = flops / (990e12 * 0.4) / 86400
    print(f"{name:>14} {N_/1e9:>8.0f}B {D_/1e12:>8.1f}T {flops:>12.2e} {days:>15.0f}")

print(f"\nGPT-3 paper reports 3.14e23. Our estimate: {6*175e9*300e9:.2e}  <- essentially exact.")

# where attention FLOPs overtake parameter FLOPs
print("\nAttention FLOPs (~12*n_layer*d_model*T^2) vs parameter FLOPs (~2*N*T):")
for name, N_, L_, C_ in [("7B", 7e9, 32, 4096), ("70B", 70e9, 80, 8192)]:
    T_cross = 2 * N_ / (12 * L_ * C_)
    print(f"  {name:>4}: crossover at T ~ {T_cross:,.0f} tokens")
print("\nBelow that, projections dominate. Above it, attention does.")
print("That is why 'attention is O(T^2)' only became urgent when contexts grew.")

---
## Self-check

1. `(4,12,128,64) @ (4,12,64,128)` -> ?
2. Why does `(10,4) + (10,)` fail while `(10,4) + (4,)` works?
3. In `'bhqd,bhkd->bhqk'`, which axis is summed?
4. Roughly how many FLOPs to train a 7B model on 2T tokens?

<details><summary>Answers</summary>

1. `(4,12,128,128)` — the 64s meet and vanish. This is the attention score matrix.
2. Broadcasting aligns from the **right**. `(10,)` lines up against the last axis
   (size 4); 10≠4 and neither is 1. Use `(10,1)` to add per-row.
3. `d` — it appears in both inputs but not the output.
4. `6 × 7e9 × 2e12 = 8.4e22`.

</details>

**Next:** `05_attention.ipynb`